# Week 6 — Day 5: Fine-tuning a Frontier Model

Use OpenAI's fine-tuning API to create a private variant of GPT-4.1-nano specialized for price prediction.

**3 steps:**
1. Format the training and validation data as JSONL and upload to OpenAI
2. Kick off a fine-tune job and monitor it
3. Evaluate the fine-tuned model against the test set

OpenAI recommends 50-100 examples to start. We use 100 — costs are minimal at this scale. The 20K-example version cost ~$3.42.

## Setup

In [ ]:
import os
import re
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items import Item
from pricer.evaluator import evaluate

In [ ]:
LITE_MODE = False
load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
username = "YOUR_HF_USERNAME"  # replace with your Hugging Face username
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
openai = OpenAI()

In [ ]:
# Start small — 100 train, 50 validation
fine_tune_train = train[:100]
fine_tune_validation = val[:50]
len(fine_tune_train)

## Step 1 — Prepare and upload JSONL files

In [ ]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

In [ ]:
messages_for(fine_tune_train[0])

In [ ]:
def make_jsonl(items):
    """Format items as JSONL — one chat-completion training example per line."""
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str + '}\n'
    return result.strip()

In [ ]:
print(make_jsonl(train[:3]))

In [ ]:
def write_jsonl(items, filename):
    with open(filename, "w") as f:
        f.write(make_jsonl(items))

In [ ]:
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")

In [ ]:
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

In [ ]:
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")
train_file

In [ ]:
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")
validation_file

## Step 2 — Kick off the fine-tune job

Monitor at https://platform.openai.com/finetune

In [ ]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="pricer"
)

In [ ]:
openai.fine_tuning.jobs.list(limit=1)

In [ ]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id
job_id

In [ ]:
openai.fine_tuning.jobs.retrieve(job_id)

In [ ]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

## Step 3 — Test the fine-tuned model

In [ ]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model
fine_tuned_model_name

In [ ]:
def test_messages_for(item):
    """At inference time, only send the user message — no assistant target."""
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [ ]:
test_messages_for(test[0])

In [ ]:
def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content

In [ ]:
print("Actual:", test[0].price)
print("Predicted:", gpt_4__1_nano_fine_tuned(test[0]))

In [ ]:
evaluate(gpt_4__1_nano_fine_tuned, test)

In [ ]:
# Comparison of fine-tune sizes from previous runs
# 96.58 — gpt-4o-mini, 200 examples
# 79.29 — gpt-4o-mini, 2000 examples
# 82.26 — gpt-4.1-nano, 2000 examples
# 67.75 — gpt-4.1-nano, 20,000 examples